# Evaluation of the hierarchy attribute extraction pipeline

Here we evaluate the performance of **Step 2** of the Ariadne pipeline: extracting SNOMED CT hierarchy attributes for medical terms.

The pipeline:
1. Retrieves similar reference SNOMED terms (few-shot examples)
2. Uses an LLM to extract attribute components from the medical term
3. Retrieves candidate concepts for each extracted component via vector search
4. Uses an LLM to select the best candidate for each attribute

## Setup

Before running this notebook, make sure the environment is set up as described in the README.
This includes:
- Credentials for a database with the OHDSI Vocabulary loaded (including `snomed_attribute` table)
- Credentials for the LLM API
- **Java 17+** installed and on PATH
- **snomed-owl-toolkit JAR** — download from [GitHub releases](https://github.com/IHTSDO/snomed-owl-toolkit/releases)
- **SNOMED CT International Edition RF2 snapshot** — obtain from [MLDS](https://mlds.ihtsdotools.org/)

All results will be stored in the `data/notebook_results` folder.
Most code blocks will load results from file if they already exist, to save time and costs.
Delete those files to rerun the corresponding steps.

In [1]:
import json
from pathlib import Path
import logging

import pandas as pd
from dotenv import load_dotenv

from ariadne.utils.config import Config

project_root = Path.cwd().parent.parent
load_dotenv()

logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(message)s")

config = Config()

## Create Attribute table

We require the `snomed_attribute` table in the database. This table contains all SNOMED concepts that can be retrieved as attribute values (e.g. finding sites, morphologies, causative agents). Use if_exists = skip to use existing tables and if_exists = rebuild to recreate the tables.

In [2]:
from ariadne.hierarchy.attribute_table_builder import build_attribute_table

build_attribute_table(
    config.hierarchy,
    if_exists="skip"
)

2026-10-28 23:03:55,259  snomed_attribute already populated — skipping.


## Gold standard

We use the hierarchy attributes gold standard file.
Each row maps a source concept to an expected SNOMED attribute (e.g. finding site, associated morphology).

This gold standard contains the following columns:
- `source_concept_id`: the OMOP concept ID of the source term (may be empty for new vocabularies)
- `source_concept_code`: the source concept code
- `source_vocabulary_id`: the source vocabulary (together with `source_concept_code` forms the primary key)
- `source_term`: the source term text
- `target_concept_id`: the concept ID of the expected attribute value (e.g. a finding site concept)
- `target_concept_code`: the concept code of the expected attribute value (e.g. a finding site concept)
- `target_concept_name`: the name of the expected attribute value
- `attribute_category`: the SNOMED relationship type (e.g. `Has finding site`, `Has asso morph`, `Has causative agent`)

In [3]:
attribute_gs_path = project_root / "data" / "gold_standards" / "hierarchy_attributes_snomed_gs.csv"
# Alternative: use unmatched terms from the exact matching pipeline (Step 1)
# attribute_gs_path = project_root / "data" / "notebook_results" / "exact_matching_vector_search_results.csv"

attribute_gs = pd.read_csv(attribute_gs_path)

# If using exact_matching_vector_search_results.csv, filter to unmatched terms only:
# attribute_gs = attribute_gs[attribute_gs["mapped_concept_id"] == -1]

print(f"Attribute GS: {len(attribute_gs)} rows")
unique_terms = attribute_gs[
    ["source_concept_id", "source_concept_code", "source_vocabulary_id", "source_term"]
].drop_duplicates()
print(f"Unique terms: {len(unique_terms)}")
attribute_gs.head(10)

Attribute GS: 776 rows
Unique terms: 275


,source_concept_id,source_concept_code,source_vocabulary_id,source_term,target_concept_id,target_concept_code,target_concept_name,attribute_category
0,22274,94899001,SNOMED,Neoplasm of uncertain behavior of larynx,4262229,4596009,Laryngeal structure,Has finding site
1,22274,94899001,SNOMED,Neoplasm of uncertain behavior of larynx,4313421,86251006,Neoplasm of uncertain behavior,Has asso morph
2,22288,191169008,SNOMED,Hereditary elliptocytosis,4030871,14089001,Red blood cell count,Has interprets
3,22288,191169008,SNOMED,Hereditary elliptocytosis,4083207,281300000,Below reference range,Has interpretation
4,22288,191169008,SNOMED,Hereditary elliptocytosis,4116830,255399007,Congenital,Has occurrence
5,22288,191169008,SNOMED,Hereditary elliptocytosis,4181412,52101004,Present,Has interpretation
6,22288,191169008,SNOMED,Hereditary elliptocytosis,4204670,308490002,Pathological developmental process,Has pathology
7,22288,191169008,SNOMED,Hereditary elliptocytosis,4218976,41898006,Erythrocyte,Has finding site
8,22288,191169008,SNOMED,Hereditary elliptocytosis,4224294,404227002,Hemolysis,Has interprets
9,22288,191169008,SNOMED,Hereditary elliptocytosis,4259404,45028007,Elliptocyte,Has asso morph


## Run the hierarchy attribute extraction pipeline

We use `process_hierarchy` from the runner module, which:
1. Iterates over each unique term in the gold standard
2. Runs the four-step attribute extraction pipeline (`extract_attributes`)
3. Checks attributes against SNOMED MRCM rules
4. Caches each successful result to a per-term JSON file, so re-running resumes and only regenerates terms that failed or were never processed


In [4]:
from ariadne.hierarchy.llm_attribute_runner import process_hierarchy
from ariadne.evaluation.hierarchy_attribute_evaluator import build_prediction_rows
from ariadne.hierarchy.searchers import SnomedAttributeSearcher, SnomedReferenceConceptVectorSearcher
from ariadne.hierarchy.mrcm import MrcmRules
from ariadne.hierarchy.mrcm_validator import validate_role_groups

# Per-term cache in data/hierarchy_attribute_responses makes re-runs free; delete it after prompt changes.
gs_concept_ids = set(attribute_gs["source_concept_id"].dropna().unique())
attr_searcher = SnomedAttributeSearcher(hierarchy_settings=config.hierarchy)
ref_searcher = SnomedReferenceConceptVectorSearcher(hierarchy_settings=config.hierarchy, exclude_concept_ids=gs_concept_ids)
results = process_hierarchy(
    unique_terms,
    attribute_searcher=attr_searcher,
    reference_searcher=ref_searcher,
    hierarchy_settings=config.hierarchy,
    source_term_column="source_term",
    source_code_column="source_concept_code",
    source_vocab_column="source_vocabulary_id",
    source_id_column="source_concept_id",
)
with open(project_root / "data" / "notebook_results" / "hierarchy_results_raw.json", "w") as f:
    json.dump(results, f, indent=2, default=str)  # debugging copy only

total_cost = sum(r.get("cost", {}).get("total_cost", 0.0) for r in results if "cost" in r)
print(f"Total API cost: ${total_cost:.4f}")

mrcm_rules = MrcmRules.load(config.hierarchy.mrcm, config.hierarchy.attributes)
violations = validate_role_groups(results, mrcm_rules)
violations.to_csv(project_root / "data" / "notebook_results" / "hierarchy_mrcm_violations.csv", index=False)
print(f"MRCM violations: {len(violations)} across {violations['source_term'].nunique() if len(violations) else 0} terms")

results_df = pd.DataFrame(build_prediction_rows(results, config.hierarchy))
results_df.to_csv(project_root / "data" / "notebook_results" / "hierarchy_attribute_results.csv", index=False)
print(f"Saved {len(results_df)} attribute rows to hierarchy_attribute_results.csv")
results_df.head()

2026-10-28 23:03:59,948  Resuming: 275 terms already cached in /Users/aostrop1/Library/CloudStorage/OneDrive-JNJ/projects/aostropolets_git/Ariadne/data/hierarchy_attribute_responses
2026-10-28 23:03:59,956  All terms already processed.


Total API cost: $6.6876
MRCM violations: 0 across 0 terms
Saved 765 attribute rows to hierarchy_attribute_results.csv


,source_key,source_concept_id,source_concept_code,source_vocabulary_id,source_term,predicted_target_concept_id,predicted_target_concept_name,predicted_target_concept_code,attribute_category,relationship_groups
0,SNOMED:10155006,73021,10155006,SNOMED,Bilateral congenital dislocation of hip,4227147,Dislocation,87642003,Has asso morph,1;2
1,SNOMED:10155006,73021,10155006,SNOMED,Bilateral congenital dislocation of hip,4177897,Right hip joint structure,362908009,Has finding site,1
2,SNOMED:10155006,73021,10155006,SNOMED,Bilateral congenital dislocation of hip,4176631,Left hip joint structure,362907004,Has finding site,2
3,SNOMED:10155006,73021,10155006,SNOMED,Bilateral congenital dislocation of hip,4204670,Pathological developmental process,308490002,Has pathology,1;2
4,SNOMED:10155006,73021,10155006,SNOMED,Bilateral congenital dislocation of hip,4116830,Congenital,255399007,Has occurrence,1;2


## Evaluation

We evaluate the pipeline results against the gold standard using a full outer join.
Each predicted attribute is matched against the expected attribute by `(source_key, target_concept_id, attribute_category)`,
where `source_key` combines `source_vocabulary_id` and `source_concept_code`.

As of now, excludes comparison of groups.

In [5]:
from ariadne.evaluation.hierarchy_attribute_evaluator import evaluate_results

# Set output directory relative to project root
output_dir = project_root / "data" / "notebook_results"

eval_df = evaluate_results(results, str(attribute_gs_path), output_dir, config.hierarchy)
eval_df.head()

2026-10-28 23:04:00,213  Gold standard rows: 778
2026-10-28 23:04:00,214  Predicted rows:     765
2026-10-28 23:04:00,214  Matched:            702
2026-10-28 23:04:00,214  Precision:          91.8%
2026-10-28 23:04:00,215  Recall:             90.2%
2026-10-28 23:04:00,215  F1:                 91.0%
2026-10-28 23:04:00,222  Combined evaluation saved: /Users/aostrop1/Library/CloudStorage/OneDrive-JNJ/projects/aostropolets_git/Ariadne/data/notebook_results/hierarchy_attribute_evaluation.csv (841 rows)


,source_key,source_concept_id,source_concept_code,source_vocabulary_id,source_term,attribute_category,gs_target_concept_id,gs_target_concept_code,gs_target_concept_name,predicted_target_concept_id,predicted_target_concept_code,predicted_target_concept_name,matched,status,relationship_groups
0,SNOMED:10155006,73021.0,10155006.0,SNOMED,Bilateral congenital dislocation of hip,Has asso morph,4227147.0,87642003.0,Dislocation,4227147,87642003,Dislocation,True,match,1;2
1,SNOMED:10155006,73021.0,10155006.0,SNOMED,Bilateral congenital dislocation of hip,Has finding site,4176631.0,362907004.0,Left hip joint structure,4176631,362907004,Left hip joint structure,True,match,2
2,SNOMED:10155006,73021.0,10155006.0,SNOMED,Bilateral congenital dislocation of hip,Has finding site,4177897.0,362908009.0,Right hip joint structure,4177897,362908009,Right hip joint structure,True,match,1
3,SNOMED:10155006,73021.0,10155006.0,SNOMED,Bilateral congenital dislocation of hip,Has occurrence,4116830.0,255399007.0,Congenital,4116830,255399007,Congenital,True,match,1;2
4,SNOMED:10155006,73021.0,10155006.0,SNOMED,Bilateral congenital dislocation of hip,Has pathology,4204670.0,308490002.0,Pathological developmental process,4204670,308490002,Pathological developmental process,True,match,1;2


### Per-category breakdown

In [6]:
breakdown = (
    eval_df.groupby("attribute_category")["status"]
    .value_counts()
    .unstack(fill_value=0)
    .reindex(columns=["match", "missed", "extra"], fill_value=0)
)
breakdown["precision%"] = (
    breakdown["match"] / (breakdown["match"] + breakdown["extra"]).clip(lower=1) * 100
).round(1)
breakdown["recall%"] = (
    breakdown["match"] / (breakdown["match"] + breakdown["missed"]).clip(lower=1) * 100
).round(1)

breakdown

status,match,missed,extra,precision%,recall%
attribute_category,,,,,
During,2,0,0,100.0,100.0
Finding asso with,3,0,0,100.0,100.0
Has asso finding,0,0,1,0.0,0.0
Has asso morph,225,14,13,94.5,94.1
Has causative agent,18,1,2,90.0,94.7
Has clinical course,28,0,2,93.3,100.0
Has due to,49,4,0,100.0,92.5
Has finding context,2,0,0,100.0,100.0
Has finding site,246,42,39,86.3,85.4


### Stated parent selection

We then find most appropriate primitive parent (N set by 'max_parents' in yaml).

For each term, we find similar standard SNOMED concepts (`num_neighbours`) and retrieve their primitive parents. A parent qualifies when has at least `min_support` (sum of similarities of neighbours sitting below a candidate parents over total similarity). Then, the deepest qualifying primitives (`min_levels_of_separation` from Clinical Finding) are chosen. When nothing qualifies, the term falls back to Clinical Finding.

For held-out SNOMED terms the source concept and its descendants are excluded from the neighbours (`exclude_source_descendants`).

In [7]:
import psycopg
from ariadne.hierarchy.parent_selector import build_stated_parents_map
from ariadne.utils.utils import get_environment_variable

conn_str = get_environment_variable("VOCAB_CONNECTION_STRING").replace("+psycopg2", "").replace("+psycopg", "")
schema = get_environment_variable("VOCAB_SCHEMA")
parent_settings = config.hierarchy.parent_selection

with open(project_root / "data" / "notebook_results" / "hierarchy_results_raw.json") as fh:
    raw_results = json.load(fh)

with psycopg.connect(conn_str) as parent_conn:
    stated_parents_map, parent_details = build_stated_parents_map(raw_results, parent_conn, schema, parent_settings)
parent_details.to_csv(project_root / "data" / "notebook_results" / "hierarchy_stated_parents.csv", index=False)

per_source = parent_details.drop_duplicates("source_concept_id")
print(f"Stated parents (num_neighbours={parent_settings.num_neighbours}, min_support={parent_settings.min_support}, "
      f"max_parents={parent_settings.max_parents}):")
print(f"  Primitive parent below Clinical finding : {(~per_source['is_fallback']).sum()}/{len(per_source)}")
print(f"  Fell back to Clinical finding           : {per_source['is_fallback'].sum()}/{len(per_source)}")
parent_details.head()

2026-10-28 23:04:02,651  Stated parents: 258 terms with a primitive parent below the root, 17 fell back to Clinical finding.


Stated parents (num_neighbours=10, min_support=0.5, max_parents=2):
  Primitive parent below Clinical finding : 258/275
  Fell back to Clinical finding           : 17/275


,source_concept_id,source_term,parent_rank,parent_concept_id,parent_concept_code,parent_concept_name,parent_depth,support,num_neighbours,is_fallback
0,73021,Bilateral congenital dislocation of hip,1,4274025,64572001,Disease,1,1.0000,10,False
1,26378,Hyperpituitarism,1,4274025,64572001,Disease,1,1.0000,10,False
2,25572,Disorder of salivary gland,1,4274025,64572001,Disease,1,0.8102,10,False
3,28356,Overlapping malignant neoplasm of major saliva...,1,4274025,64572001,Disease,1,1.0000,10,False
4,25748,Overlapping malignant neoplasm of esophagus,1,4274025,64572001,Disease,1,1.0000,10,False


In [8]:
# Evaluation: stated parents vs. the held-out concepts' proximal primitive parents fetched from OHDSI Vocabularies
from ariadne.evaluation.stated_parent_evaluator import OUTCOMES, evaluate_stated_parents

with psycopg.connect(conn_str) as parent_conn:
    parent_eval = evaluate_stated_parents(parent_details, parent_conn, schema, parent_settings)
parent_eval.to_csv(project_root / "data" / "notebook_results" / "hierarchy_stated_parent_evaluation.csv", index=False)

counts = parent_eval["outcome"].value_counts().reindex(OUTCOMES, fill_value=0)
print(pd.DataFrame({"count": counts, "%": (counts / len(parent_eval) * 100).round(1)}).to_string())
parent_eval[parent_eval["outcome"] == "wrong"].head()

              count     %
outcome                  
exact           199  72.4
partial          19   6.9
too_general      40  14.5
wrong            14   5.1
no_reference      3   1.1


,source_concept_id,source_term,selected_parents,reference_parents,outcome
26,609115,Proliferative retinopathy of left eye due to d...,Nonproliferative retinopathy due to diabetes m...,Proliferative retinopathy,wrong
36,26908,Vincent's angina,Non-plaque induced gingival disease; Traumatic...,Traumatic or non-traumatic injury; Sputum finding,wrong
42,72857,Late effect of rickets,Rickets,Disease,wrong
43,22288,Hereditary elliptocytosis,Anemia due to intrinsic red cell abnormality; ...,Hereditary hemolytic anemia; Erythrocyte membr...,wrong
63,74739,Osteopathy from poliomyelitis,Neuromuscular scoliosis,Disease,wrong


## RF2 Delta Export

Convert the predicted attributes to SNOMED CT RF2 delta files for use with the ELK reasoner.

For SNOMED held-out set, source OMOP concept IDs are remapped to synthetic SCTIDs (starting at 1 000 000 001).

Concept definitions are expressed as **OWL Functional Syntax axioms** in the OWL Axiom Reference Set:

- A concept with attributes becomes `EquivalentClasses(:src ObjectIntersectionOf(:parent ObjectSomeValuesFrom(:609096000 ...)))`
- A concept with no attributes falls back to `SubClassOf(:src :parent)`

The output ZIP contains files under `Delta/`:
- `Delta/Terminology/sct2_Concept_Delta_INT_{date}.txt` — one row per source term (sufficiently defined)
- `Delta/Terminology/sct2_StatedRelationship_Delta_INT_{date}.txt` — header-only placeholder
- `Delta/Terminology/sct2_Relationship_Delta_INT_{date}.txt` — header-only placeholder (required by toolkit)
- `Delta/Refset/Content/der2_sRefset_OWLAxiomDelta_INT_{date}.txt` — OWL axiom per source concept
- `Delta/Refset/Metadata/der2_ssRefset_ModuleDependencyDelta_INT_{date}.txt` — module dependency

In [9]:
import sys, shutil

# Force a fresh import — drop ALL ariadne.hierarchy modules from cache
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith("ariadne"):
        del sys.modules[mod_name]

from ariadne.hierarchy.rf2_exporter import export_to_rf2

rf2_output_dir = project_root / "data" / "rf2_output"
# Clean previous delta files to avoid stale artifacts
old_delta = rf2_output_dir / "SnomedCT_YourDelta"
if old_delta.exists():
    shutil.rmtree(old_delta)
old_delta2 = rf2_output_dir / "Delta"
if old_delta2.exists():
    shutil.rmtree(old_delta2)

zip_path, id_map = export_to_rf2(
    source=project_root / "data" / "notebook_results" / "hierarchy_attribute_results.csv",
    output_dir=rf2_output_dir,
    hierarchy_settings=config.hierarchy,
    stated_parent=stated_parents_map,  
)
print(f"RF2 delta written to: {zip_path}")

# Verify OWL content inline
import zipfile
with zipfile.ZipFile(zip_path) as zf:
    owl_name = next(n for n in zf.namelist() if "OWLAxiom" in n)
    with zf.open(owl_name) as f:
        owl_lines = f.read().decode("utf-8").splitlines()
print(f"OWL axiom rows: {len(owl_lines) - 1}")
if owl_lines[1:]:
    sample = owl_lines[1].split("\t")
    print(f"Sample expression: {sample[-1][:150]}")
id_map.head()


2026-10-28 23:04:05,183  Exporting 765 attribute rows to RF2 delta (date=20261028)…
2026-10-28 23:04:05,189    ID mapping: 272 concepts → id_mapping_20261028.csv
2026-10-28 23:04:05,190    Concept file: 272 concepts → sct2_Concept_Delta_INT_20261028.txt
2026-10-28 23:04:05,192    StatedRelationship file (header-only): sct2_StatedRelationship_Delta_INT_20261028.txt
2026-10-28 23:04:05,192    Relationship file (header-only placeholder): sct2_Relationship_Delta_INT_20261028.txt
2026-10-28 23:04:05,196    OWL Axiom refset: 272 concepts (272 with attributes, 0 attr-only Is a) → der2_sRefset_OWLAxiomDelta_INT_20261028.txt
2026-10-28 23:04:05,199    Module Dependency refset: der2_ssRefset_ModuleDependencyDelta_INT_20261028.txt
2026-10-28 23:04:05,208  RF2 delta ZIP: /Users/aostrop1/Library/CloudStorage/OneDrive-JNJ/projects/aostropolets_git/Ariadne/data/rf2_output/snomed_delta_20261028.zip


RF2 delta written to: /Users/aostrop1/Library/CloudStorage/OneDrive-JNJ/projects/aostropolets_git/Ariadne/data/rf2_output/snomed_delta_20261028.zip
OWL axiom rows: 272
Sample expression: EquivalentClasses(:1000000001 ObjectIntersectionOf(:64572001 ObjectSomeValuesFrom(:609096000 ObjectIntersectionOf(ObjectSomeValuesFrom(:116676008 :876


,omop_concept_id,synthetic_sctid,concept_name
0,73021,1000000001,Bilateral congenital dislocation of hip
1,26378,1000000002,Hyperpituitarism
2,25572,1000000003,Disorder of salivary gland
3,28356,1000000004,Overlapping malignant neoplasm of major saliva...
4,25748,1000000005,Overlapping malignant neoplasm of esophagus


## SNOMED Classification (ELK Reasoner)

Run classifier ([snomed-owl-toolkit](https://github.com/IHTSDO/snomed-owl-toolkit)) that takes the RF2 delta ZIP produced above and a base SNOMED CT 
snapshot, converts both to OWL, runs ELK, and returns the inferred parent relationships.

### Pre-classification validation

Check the RF2 delta for structural issues before sending it to the classifier.

In [10]:
from ariadne.hierarchy.classifier import (
    classify_delta,
    parse_classification_results,
    pre_classification_checks
)

delta_zip = project_root / "data" / "rf2_output" / next(
    f.name for f in (project_root / "data" / "rf2_output").iterdir()
    if f.name.startswith("snomed_delta_") and f.name.endswith(".zip")
)

issues = pre_classification_checks(delta_zip, config.hierarchy)
if issues:
    print("Issues found:")
    for issue in issues:
        print(f"  - {issue}")
else:
    print("All pre-classification checks passed.")

All pre-classification checks passed.


### Run classification

The snomed-owl-toolkit converts RF2 → OWL, runs ELK, and produces a results ZIP
with the inferred "Is a" relationships.

In [11]:
results_zip = classify_delta(
    delta_zip,
    base_snomed_zip=str(project_root / "data" / "SnomedCT_ManagedServiceUS_PRODUCTION_US1000124_20260301T120000Z.zip"),
    toolkit_jar=str(project_root / "tools" / "snomed-owl-toolkit-5.3.0-executable.jar"),
    output_dir=project_root / "data" / "rf2_output",
)
print(f"Classification results: {results_zip}")

2026-10-28 23:04:05,234  Running classification: java -Xms4g --add-opens java.base/java.lang=ALL-UNNAMED -jar /Users/aostrop1/Library/CloudStorage/OneDrive-JNJ/projects/aostropolets_git/Ariadne/tools/snomed-owl-toolkit-5.3.0-executable.jar -classify -rf2-snapshot-archives /Users/aostrop1/Library/CloudStorage/OneDrive-JNJ/projects/aostropolets_git/Ariadne/data/SnomedCT_ManagedServiceUS_PRODUCTION_US1000124_20260301T120000Z.zip -rf2-authoring-delta-archive /Users/aostrop1/Library/CloudStorage/OneDrive-JNJ/projects/aostropolets_git/Ariadne/data/rf2_output/snomed_delta_20261028.zip
2026-10-28 23:05:29,351  snomed-owl-toolkit stdout:
2026-10-28 23:04:05,724 [INFO  ] [main] org.snomed.otf.owltoolkit.util.TimerUtil - Timer Classification: started
2026-10-28 23:04:05,724 [INFO  ] [main] org.snomed.otf.owltoolkit.service.SnomedReasonerService - Checking requested reasoner is available
2026-10-28 23:04:05,726 [INFO  ] [main] org.snomed.otf.owltoolkit.util.TimerUtil - Timer Classification: Create

Classification results: /Users/aostrop1/Library/CloudStorage/OneDrive-JNJ/projects/aostropolets_git/Ariadne/data/rf2_output/classification-results-2026-10-28_23-04-05.zip


### Parse results and resolve names

Extract the new inferred "Is a" relationships and resolve SNOMED concept names.

In [12]:

new_is_a, removed, equiv_df = parse_classification_results(results_zip)

print(f"New inferred 'Is a' relationships: {len(new_is_a)}")
print(f"Redundant relationships removed:   {len(removed)}")
print(f"Equivalent concept rows:           {len(equiv_df)}")

# Build lookup structures
synth_to_omop = dict(zip(
    id_map["synthetic_sctid"].astype(str),
    id_map["omop_concept_id"].astype(str),
))
synth_ids = set(id_map["synthetic_sctid"].astype(str))

# src_name_map: source ID → concept name (from id_map + attribute_gs override)
src_name_map = {}
for _, m in id_map.iterrows():
    omop_str = str(m["omop_concept_id"])
    if str(m.get("concept_name", "")):
        src_name_map[omop_str] = str(m["concept_name"])
for _, gs_row in attribute_gs.iterrows():
    sid = gs_row["source_concept_id"]
    key = str(sid) if pd.notna(sid) else f"{gs_row['source_vocabulary_id']}:{gs_row['source_concept_code']}"
    src_name_map[key] = gs_row["source_term"]

# Collect all real SCTIDs that appear as ELK results (parents or children)
real_sctids = set()
for _, row in new_is_a.iterrows():
    src, dest = str(row["sourceId"]), str(row["destinationId"])
    src_is_synth = src in synth_ids
    dest_is_synth = dest in synth_ids
    if src_is_synth and not dest_is_synth:
        real_sctids.add(dest)
    elif not src_is_synth and dest_is_synth:
        real_sctids.add(src)

# Resolve SCTID → concept_name for display (keep SCTID as the ID)
sctid_name_map: dict[str, str] = {}
if real_sctids:
    with psycopg.connect(conn_str) as conn:
        with conn.cursor() as cur:
            cur.execute(
                f"""
                SELECT concept_code, concept_name
                FROM {schema}.concept
                WHERE vocabulary_id = 'SNOMED'
                  AND concept_code = ANY(%s)
                """,
                (list(real_sctids),),
            )
            for code, name in cur.fetchall():
                sctid_name_map[str(code)] = name

unmapped = real_sctids - set(sctid_name_map)
if unmapped:
    print(f"{len(unmapped)} SCTIDs not found in CONCEPT table (will keep SCTID as fallback)")

# Build equivalence pairs from the ELK equivalences refset
# Group by mapTarget UUID — concepts sharing the same group are equivalent
from collections import defaultdict
equiv_groups: dict[str, list[str]] = defaultdict(list)
for _, row in equiv_df.iterrows():
    sctid = str(row["referencedComponentId"])
    group = str(row["mapTarget"])
    equiv_groups[group].append(sctid)

# Generate "Maps to" pairs from equivalence groups (all pairwise within group)
equiv_pairs_set: set[tuple[str, str]] = set()
for group_members in equiv_groups.values():
    # Only keep pairs where at least one member is a synthetic (delta) concept
    synth_members = [m for m in group_members if m in synth_ids]
    if not synth_members:
        continue
    for i, a in enumerate(group_members):
        for b in group_members[i+1:]:
            if a in synth_ids or b in synth_ids:
                equiv_pairs_set.add((min(a, b), max(a, b)))

print(f"Equivalence groups: {len(equiv_groups)}, pairs involving delta concepts: {len(equiv_pairs_set)}")

# Build parents_df
rows = []
skipped_base_only = 0
for _, row in new_is_a.iterrows():
    src, dest = str(row["sourceId"]), str(row["destinationId"])
    src_is_synth = src in synth_ids
    dest_is_synth = dest in synth_ids

    if src_is_synth and dest_is_synth:
        continue  # inter-delta hierarchy handled via equivalences
    if not src_is_synth and not dest_is_synth:
        skipped_base_only += 1
        continue

    if src_is_synth and not dest_is_synth:
        omop_id = synth_to_omop[src]
        rows.append({
            "source_concept_id":   omop_id,
            "source_term": src_name_map.get(omop_id, omop_id),
            "relationship_id": "Is a",
            "target_concept_code": dest,
            "target_concept_name": sctid_name_map.get(dest, dest),
        })
    else:
        omop_id = synth_to_omop[dest]
        rows.append({
            "source_concept_id":   omop_id,
            "source_term": src_name_map.get(omop_id, omop_id),
            "relationship_id": "Subsumes",
            "target_concept_code": src,
            "target_concept_name": sctid_name_map.get(src, src),
        })

# Add equivalent pairs as "Maps to"
for a, b in equiv_pairs_set:
    a_is_synth = a in synth_ids
    b_is_synth = b in synth_ids
    if a_is_synth and b_is_synth:
        # Both are delta concepts → use source IDs
        omop_a = synth_to_omop[a]
        omop_b = synth_to_omop[b]
        rows.append({
            "source_concept_id":   omop_a,
            "source_term": src_name_map.get(omop_a, omop_a),
            "relationship_id": "Maps to",
            "target_concept_code": omop_b,
            "target_concept_name": src_name_map.get(omop_b, omop_b),
        })
    elif a_is_synth:
        # a is delta, b is real SNOMED
        omop_a = synth_to_omop[a]
        rows.append({
            "source_concept_id":   omop_a,
            "source_term": src_name_map.get(omop_a, omop_a),
            "relationship_id": "Maps to",
            "target_concept_code": b,
            "target_concept_name": sctid_name_map.get(b, b),
        })
    else:
        # b is delta, a is real SNOMED
        omop_b = synth_to_omop[b]
        rows.append({
            "source_concept_id":   omop_b,
            "source_term": src_name_map.get(omop_b, omop_b),
            "relationship_id": "Maps to",
            "target_concept_code": a,
            "target_concept_name": sctid_name_map.get(a, a),
        })

parents_df = pd.DataFrame(rows, columns=[
    "source_concept_id", "source_term", "relationship_id",
    "target_concept_code", "target_concept_name",
])

parents_csv = project_root / "data" / "notebook_results" / "hierarchy_classification_parents.csv"
parents_df.to_csv(parents_csv, index=False)

n_maps_to = (parents_df["relationship_id"] == "Maps to").sum()
print(f"Saved {len(parents_df)} parent relationships → hierarchy_classification_parents.csv")
print(f"  'Is a'     : {(parents_df['relationship_id']=='Is a').sum()}")
print(f"  'Subsumes' : {(parents_df['relationship_id']=='Subsumes').sum()}")
print(f"  'Maps to'  : {n_maps_to}")
parents_df.head(20)

2026-10-28 23:05:29,435  EQUIVALENT CONCEPTS FOUND (378 rows)!  This usually indicates a modelling error — two concepts have identical defining attributes.
2026-10-28 23:05:29,458  Classification results: 1516 new 'Is a' relationships, 1115 redundant removals, 378 equivalent concept rows.


New inferred 'Is a' relationships: 1516
Redundant relationships removed:   1115
Equivalent concept rows:           378
48 SCTIDs not found in CONCEPT table (will keep SCTID as fallback)
Equivalence groups: 187, pairs involving delta concepts: 198
Saved 1691 parent relationships → hierarchy_classification_parents.csv
  'Is a'     : 502
  'Subsumes' : 991
  'Maps to'  : 198


,source_concept_id,source_term,relationship_id,target_concept_code,target_concept_name
0,73555,Transient arthropathy of the pelvic region and...,Is a,66191007,Transient arthropathy
1,72404,Joint stiffness,Subsumes,202510005,Multiple stiff joints
2,31057,Disorder of pharynx,Subsumes,2129002,Edema of pharynx
3,27835,Neoplasm of uncertain behavior of pituitary gland,Subsumes,608817003,Pituicytoma
4,30770,Chronic peptic ulcer with hemorrhage but witho...,Is a,49232000,Chronic peptic ulcer with hemorrhage
5,30770,Chronic peptic ulcer with hemorrhage but witho...,Is a,128292002,Chronic disease of cardiovascular system
6,30770,Chronic peptic ulcer with hemorrhage but witho...,Is a,1142281007,1142281007
7,30770,Chronic peptic ulcer with hemorrhage but witho...,Is a,405539004,Hemorrhage of blood vessel
8,72745,Open dislocation of elbow,Subsumes,208806005,Open traumatic subluxation of elbow
9,24134,Neck pain,Subsumes,417347005,Cervical somatic dysfunction


## Evaluation: inferred parents vs parent gold standard

Compare the ELK-inferred "Is a" parents against the actual direct "Is a" relationships
from the parent gold standard (`hierarchy_snomed_gs.csv`)

In [ ]:
# ── Full per-concept evaluation file ──────────────────────────────────────
def _join(df: pd.DataFrame, sid: str, column: str) -> str:
    values = df.loc[df["source_concept_id"] == sid, column].astype(str).drop_duplicates()
    return ";".join(sorted(values))

source_term_map = {str(r["source_concept_id"]): r["source_term"] for _, r in attribute_gs.iterrows()}
eval_rows = []
for sid in sorted(source_ids):
    gs_codes = {code for s, code in actual_set if s == sid}
    pred_codes = {code for s, code in predicted_is_a if s == sid}
    matched_codes = gs_codes & pred_codes
    is_a_itself = sid in is_a_to_itself_concepts
    if sid in maps_to_correct_concepts:
        outcome = "maps_to_correct"
    elif sid in maps_to_incorrect_concepts:
        outcome = "maps_to_incorrect"
    elif is_a_itself:
        outcome = "is_a_to_itself"
    elif matched_codes:
        outcome = "is_a_correct_parent"
    elif pred_codes:
        outcome = "is_a_incorrect"
    elif sid in subsumes_concepts:
        outcome = "subsumes_only"
    else:
        outcome = "no_relationship"
    eval_rows.append({
        "source_concept_id": sid,
        "source_term": source_term_map.get(sid, ""),
        "source_concept_code": source_code_map.get(sid, ""),
        "gs_parent_codes": ";".join(sorted(gs_codes)),
        "gs_parent_names": _join(actual_gs, sid, "target_concept_name"),
        "predicted_parent_codes": ";".join(sorted(pred_codes)),
        "predicted_parent_names": _join(is_a_df, sid, "target_concept_name"),
        "matched_parent_codes": ";".join(sorted(matched_codes)),
        "maps_to_codes": _join(maps_to_df, sid, "target_concept_code"),
        "maps_to_names": _join(maps_to_df, sid, "target_concept_name"),
        "subsumes_codes": _join(subsumes_df, sid, "target_concept_code"),
        "subsumes_names": _join(subsumes_df, sid, "target_concept_name"),
        "n_gs_parents": len(gs_codes),
        "n_predicted_parents": len(pred_codes),
        "n_matched_parents": len(matched_codes),
        "is_maps_to": sid in maps_to_concepts,
        "is_maps_to_correct": sid in maps_to_correct_concepts,
        "is_a_to_itself": is_a_itself,
        "is_a_correct_parent": bool(matched_codes),
        "has_is_a": bool(pred_codes),
        "has_subsumes": sid in subsumes_concepts,
        "outcome": outcome,
    })

classification_eval = pd.DataFrame(eval_rows)
classification_eval.to_csv(project_root / "data" / "notebook_results" / "hierarchy_classification_evaluation.csv", index=False)
print(f"\nSaved {len(classification_eval)} concept rows to hierarchy_classification_evaluation.csv")
print(classification_eval["outcome"].value_counts().to_string())
classification_eval.head()


Saved 275 concept rows to hierarchy_classification_evaluation.csv
outcome
maps_to_correct        171
is_a_correct_parent     31
is_a_to_itself          28
is_a_incorrect          21
maps_to_incorrect       20
no_relationship          3
subsumes_only            1


,source_concept_id,source_term,source_concept_code,gs_parent_codes,gs_parent_names,predicted_parent_codes,predicted_parent_names,matched_parent_codes,maps_to_codes,maps_to_names,...,n_gs_parents,n_predicted_parents,n_matched_parents,is_maps_to,is_maps_to_correct,is_a_to_itself,is_a_correct_parent,has_is_a,has_subsumes,outcome
0,193782,End-stage renal disease,46177005,153611000112104;90688005,Chronic renal failure;End stage organ failure,128290005;236423003,Chronic disease of genitourinary system;Renal ...,,709044004,Chronic kidney disease,...,2,2,0,True,False,False,False,True,True,maps_to_incorrect
1,22274,Neoplasm of uncertain behavior of larynx,94899001,126692004;94962000;95158003,Neoplasm of larynx;Neoplasm of uncertain behav...,248595008;94899001,Neoplasm of uncertain behavior of larynx;Sputu...,,,,...,3,2,0,False,False,True,False,True,False,is_a_to_itself
2,22288,Hereditary elliptocytosis,191169008,234409003;276654001;363070008;38911009;42601008,Congenital hemolytic anemia;Congenital malform...,276654001;323666000;363070008;414394009;42601008,Anemia due to intrinsic red cell abnormality;C...,276654001;363070008;42601008,,,...,5,5,3,False,False,False,True,True,True,is_a_correct_parent
3,22350,Edema of larynx,51599000,118654009;60600009,Disorder characterized by edema;Disorder of th...,51599000,Edema of larynx,,51599000,Edema of larynx,...,2,1,0,True,True,True,False,True,False,maps_to_correct
4,22492,Foreign body in pharynx,25479004,128518001;33334006;75860007,Disorder of pharynx;Foreign body in digestive ...,25479004,Foreign body in pharynx,,25479004,Foreign body in pharynx,...,3,1,0,True,True,True,False,True,False,maps_to_correct
